# CountYOLO v4 — Kaggle Training Notebook

**Yêu cầu cài trước khi chạy:**
1. Notebook Settings → **Internet: ON**
2. Notebook Settings → **Accelerator: GPU T4 x2** (hoặc P100)
3. Đã upload dataset FSC-147 lên Kaggle Dataset (xem hướng dẫn bên dưới)

**Cấu trúc dữ liệu trên Kaggle Dataset:**
```
kaggle_dataset/
└── fsc147/
    ├── images_384_VarV2/
    ├── annotation_FSC147_384.json
    └── Train_Test_Val_FSC_147.json
```
→ Sau khi Add dataset vào notebook, data sẽ nằm tại `/kaggle/input/<tên-dataset>/`

In [ ]:
# Bước 1: Kiểm tra GPU
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'GPU {i}: {props.name} — {props.total_memory / 1024**3:.1f} GB VRAM')

# Kiểm tra storage
import shutil
total, used, free = shutil.disk_usage('/kaggle/working')
print(f'\n/kaggle/working — Free: {free // 1024**3} GB / {total // 1024**3} GB')

In [ ]:
# Bước 2: Clone CountYOLO từ GitHub
import os

REPO_DIR = '/kaggle/working/countyolo'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Thien-Dan/countyolo.git {REPO_DIR}
else:
    print('Repo đã có, pull latest...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# Bước 3: Cài đặt dependencies
!pip install -q transformers scipy wandb tqdm pyyaml pillow opencv-python
!pip install -q git+https://github.com/facebookresearch/sam2.git
print('\nDependencies installed ✅')

In [ ]:
# Bước 4: Tìm và Symlink dataset FSC-147 từ /kaggle/input/
import os, glob

# Tự động tìm dataset trong /kaggle/input/
annotation_files = glob.glob('/kaggle/input/**/annotation_FSC147_384.json', recursive=True)

if annotation_files:
    # Lấy thư mục chứa annotation → đó là root FSC147
    FSC147_SRC = os.path.dirname(annotation_files[0])
    print(f'Tìm thấy FSC-147 tại: {FSC147_SRC}')
    
    # Symlink vào data/FSC147
    FSC147_DST = f'{REPO_DIR}/data/FSC147'
    os.makedirs(f'{REPO_DIR}/data', exist_ok=True)
    
    if not os.path.exists(FSC147_DST):
        os.symlink(FSC147_SRC, FSC147_DST)
        print(f'Symlinked: {FSC147_SRC} → {FSC147_DST}')
    else:
        print('Symlink đã tồn tại')
        
    # Kiểm tra nhanh
    n_images = len(os.listdir(f'{FSC147_DST}/images_384_VarV2'))
    print(f'Số ảnh FSC-147: {n_images}')
else:
    print('⚠️  KHÔNG TÌM THẤY FSC-147!')
    print('Hãy Add Dataset vào notebook:')
    print('  1. Click "+ Add data" (góc phải trên)')
    print('  2. Tìm dataset FSC-147 bạn đã upload')
    print('  3. Restart kernel và chạy lại')

In [ ]:
# Bước 5: Sanity Check — Xác nhận pipeline chạy được
!python train.py --sanity-check

In [ ]:
# Bước 6: Test LLM Inference path
!python tests/test_llm_inference.py

In [ ]:
# Bước 7: Train thật!
# Checkpoint sẽ được lưu vào /kaggle/working/countyolo/checkpoint_epoch_*.pth
# Sau khi Kaggle session kết thúc, bạn có thể download checkpoint từ Output tab

import subprocess, sys

print('Bắt đầu training...')
print('Checkpoint sẽ lưu tại:', REPO_DIR)
print('Theo dõi progress qua tqdm progress bar bên dưới\n')

!python train.py --config configs/countyolo_fsc147.yaml

In [ ]:
# Bước 8 (Optional): Lưu checkpoint tốt nhất vào /kaggle/working để download
import shutil, glob

checkpoints = sorted(glob.glob(f'{REPO_DIR}/checkpoint_epoch_*.pth'))
if checkpoints:
    latest = checkpoints[-1]
    dst = f'/kaggle/working/{os.path.basename(latest)}'
    shutil.copy(latest, dst)
    print(f'Checkpoint saved to: {dst}')
    print('Bạn có thể download từ Kaggle Output tab!')
else:
    print('Không tìm thấy checkpoint nào')